# NB_Silver_Products — Bronze to Silver Transformation (Collaborative Review & Upgrade)

## Section 1 — Purpose, Business Context & Collaborative Review Legend

- **Source Table:** `Cloud_Retail_Data_Lakehouse_v2.dbo.products` → ADLS Gen2 / OneLake Shortcut (`Files/bronze/catalog/products/`).
- **Optional Enrichment Source:** `Files/bronze/catalog/product_category_name_translation/` (`71` Portuguese → English category mappings).
- **Silver Target:** `silver.products` (Managed Delta Table) + `silver._quarantine_products` (Conditional Quarantine).
- **Grain:** One row per product — primary key `product_id` (`32,951` rows; `100%` referenced by `silver.order_items` with `0` orphans and `0` unordered products).
- **Verified Bronze Profile (`32,951` rows × `9` columns):**
  - `0` duplicate `product_id`s, `0` invalid 32-char hex IDs.
  - **Olist Source Column Typos:** `product_name_lenght` and `product_description_lenght` → renamed in Silver to `product_name_length` and `product_description_length`.
  - **Missing Category & Content Metadata (`610` rows):** `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty` are all `NULL` together on `610` products (referenced by `1,603` rows in `order_items`).
  - **Untranslated Categories (`13` products / `2` categories):** `pc_gamer` (`3` products, `9` items) and `portateis_cozinha_e_preparadores_de_alimentos` (`10` products, `15` items) exist in `products` (`73` distinct categories) but are missing from the `71`-row translation file.
  - **Physical Dimensions & Weight Anomalies (`6` products total):** `2` rows have `NULL` weight & dimensions; `4` `cama_mesa_banho` products (`30×25×30 cm`) have `product_weight_g = 0.00` (flagged & nulled so weight averages are not skewed).

### Collaborative Code-Review Legend
This notebook preserves the teammate's `product_trasformation.ipynb` logic alongside architectural upgrades:
- **`[KEPT / BUILT UPON]`**: Strong work from the original notebook (native Spark Parquet read, profiling `0`-weight products, fixing `lenght` typos, computing `product_volume_cm3`, and mapping the `2` missing English categories).
- **`[ORIGINAL — COMMENTED FOR REVIEW]`**: Original teammate code preserved inside comments (inactive so `Run All` never fails) with clear explanations of why specific patterns (`coalesce(..., 0)` on missing metadata, `double` instead of `decimal`, `dim_products` in `dbo`, and duplicate `Files/silver/catalog/products` Parquet export) needed adjustment.
- **`[IMPROVED REPLACEMENT]`**: Production PySpark replacement directly beneath the commented cell.
- **`[NEW ADDITION — PRODUCTION STANDARD]`**: Added enterprise controls (`WRITE_ENABLED = False`, relative paths, `Quarantine` routing, boolean quality flags, full lineage, and reconciliation).

> **WARNING — SAFETY LOCK ENABLED (`WRITE_ENABLED = False`):** Do not enable writes until a Dry Run completes cleanly and reconciliations pass.

## Section 2 — Configuration & Safety Controls `[NEW ADDITION — PRODUCTION STANDARD]`

Replaces hardcoded `abfss://` workspace/lakehouse GUIDs with relative OneLake shortcut paths (`Files/bronze/...`), sets canonical target `silver.products`, and locks writes behind `WRITE_ENABLED = False`.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Section 2: Configuration & Safety Controls
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

WRITE_ENABLED = False          # SAFETY LOCK: flip to True only after Dry Run review
RUN_ORDER_ITEM_FK_CHECK = True # Cross-check coverage against silver.order_items when present

BRONZE_PRODUCTS_PATH = "Files/bronze/catalog/products/products_20261121_013250_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"
BRONZE_TRANSLATION_PATH = "Files/bronze/catalog/product_category_name_translation/product_category_name_translation_20261121_013318_b0e93f82-84a0-4668-9a8c-9ae1da7dd183.parquet"

SILVER_SCHEMA = "silver"
SILVER_TARGET_TABLE = f"{SILVER_SCHEMA}.products"
QUARANTINE_TARGET_TABLE = f"{SILVER_SCHEMA}._quarantine_products"
SILVER_ORDER_ITEMS_TABLE = f"{SILVER_SCHEMA}.order_items"

INGESTION_RUN_ID = "b0e93f82-84a0-4668-9a8c-9ae1da7dd183"
SOURCE_SYSTEM = "sqlserver_olist"
SOURCE_TABLE = "dbo.products"

EXPECTED_BRONZE_COLS = [
    "product_id",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

print(f"Configured BRONZE_PRODUCTS_PATH    : {BRONZE_PRODUCTS_PATH}")
print(f"Configured BRONZE_TRANSLATION_PATH : {BRONZE_TRANSLATION_PATH}")
print(f"Configured SILVER_TARGET           : {SILVER_TARGET_TABLE}")
print(f"Configured QUARANTINE_TARGET       : {QUARANTINE_TARGET_TABLE}")
print(f"WRITE_ENABLED                      : {WRITE_ENABLED}")

## Section 3 — Distributed Bronze Parquet Ingestion `[KEPT / BUILT UPON]`

Keeps the teammate's clean native Spark Parquet read for both `products` and `product_category_name_translation`, upgrading only the hardcoded `abfss://` URI to the portable relative `Files/bronze/...` shortcut path and adding `_source_file` lineage.

In [ ]:
# [KEPT / BUILT UPON] Section 3: Native Spark Parquet Ingestion
df_bronze_raw = (
    spark.read
    .format("parquet")
    .load(BRONZE_PRODUCTS_PATH)
    .withColumn("_source_file", F.input_file_name())
)

df_translation_raw = spark.read.format("parquet").load(BRONZE_TRANSLATION_PATH)

bronze_count = df_bronze_raw.count()
translation_count = df_translation_raw.count()
print(f"Bronze Products Row Count    : {bronze_count:,}")
print(f"Bronze Translation Row Count : {translation_count:,}")
df_bronze_raw.printSchema()
display(df_bronze_raw.limit(5))

## Stage 1 — Schema Contract & Data Quality Profiling `[KEPT / BUILT UPON]`

Expands the teammate's Cell [2] profiling (null counts and zero/negative weight/dimensions) with formal schema-contract validation (`EXPECTED_BRONZE_COLS`) and schema-drift detection.

In [ ]:
# [KEPT / BUILT UPON] Stage 1: Schema Contract & Baseline Profiling
actual_data_cols = [c for c in df_bronze_raw.columns if c != "_source_file"]
missing_cols = [c for c in EXPECTED_BRONZE_COLS if c not in actual_data_cols]
extra_cols = [c for c in actual_data_cols if c not in EXPECTED_BRONZE_COLS]

if missing_cols:
    raise ValueError(f"Stage 1 FAILED — Missing required Bronze columns: {missing_cols}")

if extra_cols:
    print(f"WARNING — Extra columns observed in Bronze (excluded from Silver contract): {extra_cols}")

df_bronze = df_bronze_raw.select(*EXPECTED_BRONZE_COLS, "_source_file")

null_counts = df_bronze.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
    for c in EXPECTED_BRONZE_COLS
])
zero_dims = df_bronze.filter(
    (F.col("product_weight_g") <= 0) |
    (F.col("product_length_cm") <= 0) |
    (F.col("product_height_cm") <= 0) |
    (F.col("product_width_cm") <= 0)
)
print(f"Stage 1 Schema Validation: PASSED | Zero/Negative Weight/Dim Rows: {zero_dims.count()}")
display(null_counts)
display(zero_dims)

## Stage 2A — Teammate Original Cleaning & Category Enrichment `[ORIGINAL — COMMENTED FOR REVIEW]`

**What was RIGHT (`[KEPT / BUILT UPON]`):**
- Discovered and translated the `2` missing categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` → `portable_kitchen_and_food_preparers`).
- Fixed Olist's `lenght` typo (`product_name_length`, `product_description_length`).
- Nullified `product_weight_g <= 0` (`4` rows) and computed `product_volume_cm3`.

**Why three details needed adjustment:**
1. **`coalesce(..., 0)` on `name_length`, `description_length`, `photos_qty` (`610` rows) and `"unknown"` on `product_category_name`:** Replacing missing metadata with `0` distorts `AVG(photos_qty)` and `AVG(description_length)` in analytics! Keeping `NULL` as `NULL` + adding `is_missing_category` is cleaner in Silver (Gold `dim_product` can display `'Unknown'` for BI slicers).
2. **Casting `decimal(12,2)` down to `double`:** Bronze Parquet already provides exact `decimal(12,2)` for weight and dimensions; we keep `decimal(12,2)` (and `decimal(14,2)` for `product_volume_cm3`).
3. **Silent `.filter(isNotNull()).dropDuplicates(["product_id"])`:** Replaced with explicit Quarantine routing and PK checks.

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELL 4 — COMMENTED FOR REVIEW]
# Good: Mapped 2 missing EN categories, fixed 'lenght' typos, nulled <=0 weight, added volume.
# Replaced: Preserved NULLs (no 0/unknown distortion), kept decimal(12,2), added Quarantine.
# ==============================================================================
# df_silver_products = (
#     df_products.alias("p")
#     .join(
#         df_translation.alias("t"),
#         F.trim(F.col("p.product_category_name")) == F.trim(F.col("t.product_category_name")),
#         "left"
#     )
#     .select(
#         F.trim(F.col("p.product_id")).alias("product_id"),
#         F.when(F.col("p.product_category_name") == "pc_gamer", "pc_gamer")
#          .when(F.col("p.product_category_name") == "portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")
#          .when(F.col("t.product_category_name_english").isNotNull(), F.trim(F.col("t.product_category_name_english")))
#          .when(F.col("p.product_category_name").isNotNull(), F.trim(F.col("p.product_category_name")))
#          .otherwise("unknown")
#          .alias("category_name_en"),
#         F.coalesce(F.trim(F.col("p.product_category_name")), F.lit("unknown")).alias("category_name_pt"),
#         F.coalesce(F.col("p.product_name_lenght"), F.lit(0)).cast("int").alias("product_name_length"),
#         F.coalesce(F.col("p.product_description_lenght"), F.lit(0)).cast("int").alias("product_description_length"),
#         F.coalesce(F.col("p.product_photos_qty"), F.lit(0)).cast("int").alias("product_photos_qty"),
#         F.when(F.col("p.product_weight_g") <= 0, None).otherwise(F.col("p.product_weight_g")).cast("double").alias("product_weight_g"),
#         F.when(F.col("p.product_length_cm") <= 0, None).otherwise(F.col("p.product_length_cm")).cast("double").alias("product_length_cm"),
#         F.when(F.col("p.product_height_cm") <= 0, None).otherwise(F.col("p.product_height_cm")).cast("double").alias("product_height_cm"),
#         F.when(F.col("p.product_width_cm") <= 0, None).otherwise(F.col("p.product_width_cm")).cast("double").alias("product_width_cm")
#     )
#     .withColumn("product_volume_cm3", F.round(F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"), 2))
#     .filter(F.col("product_id").isNotNull())
#     .dropDuplicates(["product_id"])
# )
print("Skipped commented original Cell 4 — executing precision-safe Silver standardization next.")

## Stage 2B — Precision-Safe Standardization, Typo Fix & Category Enrichment `[IMPROVED REPLACEMENT]`

1. Normalizes `product_id` and `product_category_name` via `F.lower(F.trim(...))` (preserving `NULL`s).
2. Left-joins `df_translation_raw` and applies the teammate's verified English backfill for `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` (`product_category_name_english`).
3. Renames `product_name_lenght` → `product_name_length` and `product_description_lenght` → `product_description_length`, preserving `NULL`s and `DECIMAL(12,2)` precision.
4. Nullifies non-positive weight/dimensions (`product_weight_g == 0` on `4` rows) **after** recording `has_zero_or_null_dimensions`, and computes `product_volume_cm3` (`DECIMAL(14,2)`).

In [ ]:
# [IMPROVED REPLACEMENT] Stage 2B: Standardization, Translation Enrichment & Exact Decimals
df_trans_clean = (
    df_translation_raw
    .select(
        F.lower(F.trim(F.col("product_category_name").cast("string"))).alias("_t_cat_pt"),
        F.lower(F.trim(F.col("product_category_name_english").cast("string"))).alias("_t_cat_en"),
    )
    .dropDuplicates(["_t_cat_pt"])
)

df_std = (
    df_bronze.alias("p")
    .withColumn("product_id", F.lower(F.trim(F.col("product_id").cast("string"))))
    .withColumn("product_category_name", F.lower(F.trim(F.col("product_category_name").cast("string"))))
    .join(df_trans_clean.alias("t"), F.col("product_category_name") == F.col("t._t_cat_pt"), "left")
    .withColumn(
        "product_category_name_english",
        F.when(F.col("product_category_name") == "pc_gamer", F.lit("pc_gamer"))
        .when(F.col("product_category_name") == "portateis_cozinha_e_preparadores_de_alimentos", F.lit("portable_kitchen_and_food_preparers"))
        .when(F.col("t._t_cat_en").isNotNull(), F.col("t._t_cat_en"))
        .otherwise(F.col("product_category_name"))
    )
    .withColumn("product_name_length", F.expr("try_cast(product_name_lenght AS INT)"))
    .withColumn("product_description_length", F.expr("try_cast(product_description_lenght AS INT)"))
    .withColumn("product_photos_qty", F.expr("try_cast(product_photos_qty AS INT)"))
    .withColumn("_raw_weight", F.expr("try_cast(product_weight_g AS DECIMAL(12,2))"))
    .withColumn("_raw_length", F.expr("try_cast(product_length_cm AS DECIMAL(12,2))"))
    .withColumn("_raw_height", F.expr("try_cast(product_height_cm AS DECIMAL(12,2))"))
    .withColumn("_raw_width", F.expr("try_cast(product_width_cm AS DECIMAL(12,2))"))
    .withColumn("is_missing_category", F.col("product_category_name").isNull())
    .withColumn(
        "is_zero_or_null_dimensions",
        F.col("_raw_weight").isNull() | (F.col("_raw_weight") <= 0) |
        F.col("_raw_length").isNull() | (F.col("_raw_length") <= 0) |
        F.col("_raw_height").isNull() | (F.col("_raw_height") <= 0) |
        F.col("_raw_width").isNull() | (F.col("_raw_width") <= 0)
    )
    .withColumn("product_weight_g", F.when(F.col("_raw_weight") > 0, F.col("_raw_weight")).otherwise(F.lit(None).cast(DecimalType(12, 2))))
    .withColumn("product_length_cm", F.when(F.col("_raw_length") > 0, F.col("_raw_length")).otherwise(F.lit(None).cast(DecimalType(12, 2))))
    .withColumn("product_height_cm", F.when(F.col("_raw_height") > 0, F.col("_raw_height")).otherwise(F.lit(None).cast(DecimalType(12, 2))))
    .withColumn("product_width_cm", F.when(F.col("_raw_width") > 0, F.col("_raw_width")).otherwise(F.lit(None).cast(DecimalType(12, 2))))
    .withColumn(
        "product_volume_cm3",
        (F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")).cast(DecimalType(14, 2))
    )
)

print(f"Stage 2B complete. Distinct PT categories: {df_std.select('product_category_name').distinct().count()}")
print(f"Distinct EN categories (excl. NULL): {df_std.filter(F.col('product_category_name_english').isNotNull()).select('product_category_name_english').distinct().count()}")

## Stage 3 & 4 — PK Uniqueness, Validation & Quarantine Routing `[NEW ADDITION — PRODUCTION STANDARD]`

1. Audits `product_id` uniqueness (`0` duplicates) and 32-char lowercase hex format (`^[0-9a-f]{32}$`).
2. Checks `silver.order_items` coverage when present (`32,951` products; `0` products never ordered).
3. Routes any invalid record to `df_quarantine` with `dq_rejection_reason` and preserves all `32,951` valid products in `df_valid`.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 3 & 4: PK Check, Validation & Quarantine
hex32_regex = r"^[0-9a-f]{32}$"
exact_dup_count = bronze_count - df_bronze.dropDuplicates(EXPECTED_BRONZE_COLS).count()

dup_pk_df = (
    df_std.filter(F.col("product_id").isNotNull())
    .groupBy("product_id").count().filter(F.col("count") > 1)
    .select("product_id").withColumn("_dup_pk", F.lit(True))
)
dup_pk_cnt = dup_pk_df.count()

df_checked = (
    df_std
    .join(dup_pk_df, on="product_id", how="left")
    .withColumn("_dup_pk", F.coalesce(F.col("_dup_pk"), F.lit(False)))
    .withColumn(
        "dq_rejection_reason",
        F.concat_ws(
            " | ",
            F.when(~F.coalesce(F.col("product_id"), F.lit("")).rlike(hex32_regex), F.lit("INVALID_PRODUCT_ID_HEX32")),
            F.when(F.col("_dup_pk"), F.lit("DUPLICATE_PRODUCT_ID_PK")),
            F.when(F.col("_raw_weight") < 0, F.lit("NEGATIVE_PRODUCT_WEIGHT")),
        )
    )
)

df_valid = df_checked.filter(F.length(F.col("dq_rejection_reason")) == 0)
df_quarantine = (
    df_checked.filter(F.length(F.col("dq_rejection_reason")) > 0)
    .select(*EXPECTED_BRONZE_COLS, "dq_rejection_reason", "_source_file")
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_count = df_valid.count()
quarantine_count = df_quarantine.count()
print(f"Exact Duplicates : {exact_dup_count} | Duplicate PKs : {dup_pk_cnt}")
print(f"Valid Products   : {valid_count:,} (Expected: 32,951)")
print(f"Quarantined Rows : {quarantine_count:,} (Expected: 0)")

if RUN_ORDER_ITEM_FK_CHECK and spark.catalog.tableExists(SILVER_ORDER_ITEMS_TABLE):
    oi_prods = spark.table(SILVER_ORDER_ITEMS_TABLE).select("product_id").distinct()
    unordered_cnt = df_valid.select("product_id").join(oi_prods, on="product_id", how="left_anti").count()
    print(f"Cross-Check vs {SILVER_ORDER_ITEMS_TABLE}: {unordered_cnt} unordered products (Expected: 0)")

## Stage 5 & 6 — Quality Flag Diagnostics, Lineage & Reconciliation `[NEW ADDITION — PRODUCTION STANDARD]`

Projects the clean `17`-column Silver contract (`product_id`, Portuguese & English categories, fixed-spelling metadata lengths, exact `DECIMAL` physical dimensions & volume, `2` boolean quality flags, and `4` standardized lineage columns) and verifies `bronze_count == silver_count + quarantine_count`.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Stage 5 & 6: Diagnostics, Lineage & Reconciliation
df_silver = (
    df_valid
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("source_table", F.lit(SOURCE_TABLE))
    .withColumn("ingestion_run_id", F.lit(INGESTION_RUN_ID))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_volume_cm3",
        "is_missing_category",
        "is_zero_or_null_dimensions",
        "source_system",
        "source_table",
        "ingestion_run_id",
        "silver_processed_at",
    )
)

silver_count = df_silver.count()
assert bronze_count == silver_count + quarantine_count, (
    f"Reconciliation mismatch: bronze={bronze_count}, silver={silver_count}, quarantine={quarantine_count}"
)

diag = df_silver.agg(
    F.sum(F.when(F.col("is_missing_category"), 1).otherwise(0)).alias("missing_cat_cnt"),
    F.sum(F.when(F.col("is_zero_or_null_dimensions"), 1).otherwise(0)).alias("zero_or_null_dim_cnt"),
    F.sum(F.when(F.col("product_weight_g") <= 0, 1).otherwise(0)).alias("remaining_nonpos_weight"),
).collect()[0]

print(f"Stage 6 Reconciliation PASSED : bronze={bronze_count:,}, silver={silver_count:,}, quarantine={quarantine_count:,}")
print(f"Missing Category Flagged      : {diag['missing_cat_cnt']} (Expected: 610)")
print(f"Zero/Null Dimensions Flagged  : {diag['zero_or_null_dim_cnt']} (Expected: 6)")
print(f"Remaining <=0 Weight          : {diag['remaining_nonpos_weight']} (Expected: 0)")
df_silver.printSchema()

## Section 5A — Teammate Original Write Cells `[ORIGINAL — COMMENTED FOR REVIEW]`

**Why the original write cells `[6]` & `[7]` are commented out and replaced below:**
1. **`saveAsTable("dim_products")` (Cell [6]):** Wrote into the default `dbo` schema with a Gold-style `dim_*` name instead of canonical **`silver.products`**.
2. **Duplicate Parquet write `Files/silver/catalog/products` (Cell [7]):** Writing a second raw Parquet copy outside `Tables/silver/` creates dual-write drift; Silver tables should live in one canonical Managed Delta Table (`silver.products`).
3. **No `WRITE_ENABLED` safety gate.**

In [ ]:
# ==============================================================================
# [ORIGINAL TEAMMATE CELLS 6 & 7 — COMMENTED FOR REVIEW]
# Replaced: Unified into a single guarded Delta table write (silver.products).
# ==============================================================================
# (
#     df_silver_products.write
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .format("delta")
#     .saveAsTable("dim_products")
# )
# df_silver_products.write.mode("overwrite").parquet("Files/silver/catalog/products")
print("Skipped commented original Cells 6 & 7 — executing guarded write to silver.products next.")

## Section 5B — Guarded Delta Write (`silver.products`) `[IMPROVED REPLACEMENT]`

Creates `silver` schema if missing and writes `silver.products` (plus `silver._quarantine_products` only if `quarantine_count > 0`) strictly when `WRITE_ENABLED = True`.

In [ ]:
# [IMPROVED REPLACEMENT] Section 5B: Guarded Delta Write into silver.products
if WRITE_ENABLED:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TARGET_TABLE)
    print(f"Wrote {silver_count:,} rows to {SILVER_TARGET_TABLE}")

    if quarantine_count > 0:
        df_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE_TARGET_TABLE)
        print(f"Wrote {quarantine_count:,} rows to {QUARANTINE_TARGET_TABLE}")
    else:
        print("Zero quarantined rows — skipping quarantine table write.")
else:
    print("WRITE_ENABLED is False — dry validation only; no Silver or Quarantine data was written.")

## Section 6 — Post-Write Delta Read-Back Verification `[NEW ADDITION — PRODUCTION STANDARD]`

Reads back `silver.products` when `WRITE_ENABLED = True` and verifies `32,951` rows, `0` duplicate `product_id`s, `0` non-positive weights, and top English categories.

In [ ]:
# [NEW ADDITION — PRODUCTION STANDARD] Section 6: Post-Write Validation
if WRITE_ENABLED:
    df_check = spark.table(SILVER_TARGET_TABLE)
    chk_count = df_check.count()
    chk_dup = df_check.groupBy("product_id").count().filter(F.col("count") > 1).count()
    chk_nonpos_w = df_check.filter(F.col("product_weight_g") <= 0).count()

    assert chk_count == silver_count, f"Row mismatch: {chk_count} vs {silver_count}"
    assert chk_dup == 0, f"Duplicate product_id found: {chk_dup}"
    assert chk_nonpos_w == 0, f"Non-positive weight found: {chk_nonpos_w}"

    print(f"Post-write validation PASSED — rows={chk_count:,}, dup_pk={chk_dup}, nonpos_weight={chk_nonpos_w}")
    display(df_check.groupBy("product_category_name_english").count().orderBy(F.desc("count")).limit(10))
else:
    print("Post-write validation skipped because WRITE_ENABLED is False.")